# Блок 8 · RAG по базе знаний Handbook

Основной источник проекта — база знаний `D:\study\handbook`.
Ноутбук собирает корпус и запускает на нём тот же поисковый
RAG-пайплайн, что и консольное приложение.

Что проверяем:

1. сборка корпуса из markdown и Word-документов;
2. дедупликация и исключение служебных/чувствительных файлов;
3. поиск по 50 контрольным вопросам;
4. Top-1 и Hit@K на большом корпусе;
5. пять экспериментов с чанками и способом поиска;
6. сохранение отчётов в `reports/handbook/`.

## 1. Импорты и путь к проекту

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.config import (
    HANDBOOK_CHECK_QUESTIONS_PATH,
    CHUNK_SIZE,
    HANDBOOK_CORPUS_PATH,
    HANDBOOK_CUSTOM_QUESTION,
    HANDBOOK_DEMO_QUESTION,
    OVERLAP,
    SEARCH_METHOD,
    HANDBOOK_SOURCE_DIR,
    HANDBOOK_STATS_PATH,
    TOP_K,
    UNKNOWN_MIN_SIMILARITY,
    UNKNOWN_QUESTION,
)
from src.answerer import NO_ANSWER
from src.assistant import RagAssistant
from src.corpus_builder import HandbookCorpusBuilder
from src.data_loader import DataLoader
from src.evaluator import SearchEvaluator
from src.experiments import ExperimentRunner

pd.set_option("display.max_colwidth", 80)
print("Project root:", PROJECT_ROOT)
print("Handbook:", HANDBOOK_SOURCE_DIR)

## 2. Сборка и загрузка корпуса

In [ ]:
if not HANDBOOK_CORPUS_PATH.exists():
    HandbookCorpusBuilder(
        source_dir=HANDBOOK_SOURCE_DIR,
        output_path=HANDBOOK_CORPUS_PATH,
        stats_path=HANDBOOK_STATS_PATH,
    ).build()

documents = DataLoader(HANDBOOK_CORPUS_PATH).load_documents()
questions = DataLoader.load_check_questions(HANDBOOK_CHECK_QUESTIONS_PATH)
stats = json.loads(HANDBOOK_STATS_PATH.read_text(encoding="utf-8"))

print("Документов:", len(documents))
print("Вопросов:", len(questions))
print("Слов:", stats["total_words"])
print("Пропущено чувствительных:", stats["skipped_sensitive"])
print("Пропущено дублей:", stats["skipped_duplicates_count"])

display(pd.DataFrame([
    {"категория": key, "документов": value}
    for key, value in stats["by_category"].items()
]))
display(pd.DataFrame([
    {
        "источник": doc["id"],
        "название": doc["title"],
        "тип": doc["kind"],
        "слов": doc["words"],
    }
    for doc in sorted(documents, key=lambda item: -item["words"])[:12]
]))

## 3. Индекс с параметрами Handbook

In [ ]:
assistant = RagAssistant(
    documents,
    chunk_size=CHUNK_SIZE,
    overlap=OVERLAP,
    search_method=SEARCH_METHOD,
    top_k=TOP_K,
)
print(
    f"CHUNK_SIZE={CHUNK_SIZE}, OVERLAP={OVERLAP}, "
    f"SEARCH_METHOD={SEARCH_METHOD}, TOP_K={TOP_K}"
)
print("Чанков:", len(assistant.records))
assert len(assistant.records) > len(documents)

## 4. Демонстрационный вопрос

In [ ]:
demo_result = assistant.ask(HANDBOOK_DEMO_QUESTION)
print("Вопрос:", HANDBOOK_DEMO_QUESTION)
display(pd.DataFrame([
    {
        "место": number,
        "источник": item["doc_id"],
        "название": item["title"],
        "сходство": round(float(item["score"]), 3),
    }
    for number, item in enumerate(demo_result["found"], 1)
]))
print(demo_result["answer"])
assert demo_result["answer"] != NO_ANSWER

## 5. Контрольный вопрос по KMeans

In [ ]:
custom_result = assistant.ask(HANDBOOK_CUSTOM_QUESTION)
print("Вопрос:", HANDBOOK_CUSTOM_QUESTION)
display(pd.DataFrame([
    {
        "место": number,
        "источник": item["doc_id"],
        "название": item["title"],
        "сходство": round(float(item["score"]), 3),
    }
    for number, item in enumerate(custom_result["found"], 1)
]))
print(custom_result["answer"])
assert custom_result["answer"] != NO_ANSWER

## 6. Вопрос без подтверждённого ответа

In [ ]:
unknown_result = assistant.ask(
    UNKNOWN_QUESTION, min_similarity=UNKNOWN_MIN_SIMILARITY
)
print("Вопрос:", UNKNOWN_QUESTION)
print("Лучшее сходство:", round(float(unknown_result["best_score"]), 3))
print("Ответ:", unknown_result["answer"])
assert unknown_result["answer"] == NO_ANSWER

## 7. Проверка поиска: Top-1 и Hit@K

In [ ]:
detail_df, summary = SearchEvaluator.check(
    assistant.retriever, questions, top_k=TOP_K
)
display(detail_df.head(10))
print(
    "Top-1:", round(summary["top1_accuracy"], 3),
    "| Hit@K:", round(summary["hit_at_k"], 3),
)
print("Всего вопросов:", len(detail_df))
print("Промахов Hit@K:", int((detail_df["hit_at_k"] == 0).sum()))
assert summary["top1_accuracy"] <= summary["hit_at_k"] + 1e-9

## 8. Пять экспериментов A–E

In [ ]:
results_df, _ = ExperimentRunner(documents, questions).run()
display(results_df)
assert results_df.shape[0] == 5
assert results_df["hit_at_k"].between(0, 1).all()

In [ ]:
labels = [name.split(":", 1)[0] for name in results_df["опыт"]]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(labels, results_df["top1"], color="#4C72B0")
axes[0].set_title("Правильный источник на первом месте")
axes[1].bar(labels, results_df["hit_at_k"], color="#55A868")
axes[1].set_title("Правильный источник среди TOP-K")

for ax in axes:
    ax.set_ylim(0, 1.02)
    ax.set_ylabel("Доля вопросов")
    ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Вывод

- Корпус Handbook: 77 документов и ~167 тыс. слов.
- В нём много пересекающихся материалов, поэтому Top-1 ограничен,
  а Hit@K остаётся высоким.
- `char_wb` устойчивее `word`; длинные чанки дают лучший Top-1.
- `top_k=5` повышает Hit@K, но добавляет шум.
- Полный текст корпуса не коммитится: он собирается локально.

## 9. Полный запуск и сохранение отчётов

Ячейка ниже перезапускает полный сценарий и сохраняет CSV, JSON, TXT,
графики и markdown-отчёт в `reports/handbook/`.

In [ ]:
from main import handbook_config
from src.pipeline import run_corpus_pipeline

run_corpus_pipeline(handbook_config())